# 06 · A RAG pipeline from scratch (no framework)

After this notebook you can build and explain every stage of a RAG system with plain Python - ingest a folder, chunk, embed, index, retrieve (hybrid), gate on confidence, build a grounded prompt with citations and an explicit "say I don't know" rule, call the model, parse and verify the citations - and evaluate retrieval and citations on a labelled question set.

**Time:** ~35 min · **Runs:** offline on CPU in ~1 min (LLM calls go to the offline stand-in unless you set a key) · **Needs:** [hybrid search and RRF](04_hybrid_search_and_rrf.ipynb), [OpenAI API basics](../08_llm_apps/01_openai_api_basics.ipynb)

## Why this matters in interviews

- "Design a RAG system over a folder of documents - show me the skeleton" (`rg00`) is increasingly a **live coding** question. Doing it without a framework proves you know what the framework does.
- The senior signals are the unglamorous parts: an **empty / low-confidence branch** before the model is called, **citations verified in code**, a **fixed eval set**, and knowing which failure mode you are looking at.
- Everything here is also the answer to "the model hallucinated - how do you debug it?": print what was retrieved, check the evidence was in it, check the prompt, then blame the model.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `rg00` | Design a RAG system over a knowledge base folder: what language, what packages, and what does the skeleton code look like? |
| `rg01` | What is RAG and why does it exist? |
| `rg25` | How do you handle empty or low-confidence retrieval? |
| `rg24` | How do you verify citations so the model cannot fabricate them? |
| `rg14` | Must the query and the documents use the same embedding model? What if they do not? |
| `rg11` | How do you evaluate a RAG system? What do you measure? |
| `rg23` | What are the main RAG failure modes, and how do you tell them apart? |
| `rg13` | What is the RAG deployment checklist you would run before going live? |
| `rg55` | What is the single biggest mistake teams make with RAG? |

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER

In [ ]:
import json
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import FancyBboxPatch
from rank_bm25 import BM25Okapi

np.random.seed(0)
pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 70)

## The shape of the system

Two pipelines that share one index. The **index-time** pipeline runs once (and again whenever a document changes); the **query-time** pipeline runs on every question.

In [ ]:
def draw_pipeline(steps, title=None):
    """Boxes joined by arrows, left to right."""
    fig, ax = plt.subplots(figsize=(min(2.3 * len(steps), 11), 1.7))
    for i, s in enumerate(steps):
        x = i * 2.3
        ax.add_patch(FancyBboxPatch((x, 0), 1.9, 1, boxstyle="round,pad=0.06", fc="#e7f0ff", ec="#3b5bdb"))
        ax.text(x + 0.95, 0.5, s, ha="center", va="center", fontsize=9)
        if i < len(steps) - 1:
            ax.annotate("", xy=(x + 2.28, 0.5), xytext=(x + 1.98, 0.5), arrowprops=dict(arrowstyle="->", lw=1.6))
    ax.set_xlim(-0.2, len(steps) * 2.3); ax.set_ylim(-0.25, 1.25); ax.axis("off")
    if title: ax.set_title(title)
    plt.show()


draw_pipeline(["load folder\n(*.md)", "chunk by\nheading", "embed\n(MiniLM)", "index\nNumPy + BM25"],
              title="INDEX TIME - once, and on every document change")
draw_pipeline(["question", "hybrid\nretrieve", "confidence\ngate", "grounded\nprompt", "LLM", "verify\ncitations"],
              title="QUERY TIME - every question")

## Step 1 - Ingest: a knowledge-base folder

The Acme knowledge base (the same 15 docs as every notebook in this section) is written to a temporary folder as real `.md` files, so the pipeline starts where a real one does: a directory.

In [ ]:
# The knowledge base: 15 short markdown docs for a fictional company, Acme Cloud (a SaaS analytics product
# that also sells an 'Edge Gateway' device). This cell is data - it is identical in every 09_rag notebook.
KB = {
"refund_policy.md": """# Refund Policy

## Eligibility
Annual plans can be refunded in full within 30 days of purchase. Monthly plans can be cancelled at any time, but the current month is not refunded. Add-on credits and professional-services hours are non-refundable.

## How to request a refund
Open the billing portal, choose the invoice and click "Request refund". Only workspace owners can request refunds.

## Processing time
Approved refunds go back to the original payment method within 5-10 business days. Bank transfers can take up to 15 business days.
""",
"shipping.md": """# Hardware Shipping

## Delivery times
Edge Gateway devices ship from our warehouses in Rotterdam and Dallas. Standard delivery takes 3-5 business days. Express delivery takes 1-2 business days and costs $25.

## Tracking
A tracking link is emailed when the order leaves the warehouse. The first carrier scan can take 24 hours to appear.

## Lost or damaged shipments
If a shipment has not arrived 10 business days after dispatch, report it within 14 days so we can open a carrier investigation. We send a free replacement when the carrier confirms the loss.
""",
"warranty.md": """# Edge Gateway Warranty

## Coverage
Every Edge Gateway has a 2-year limited hardware warranty from the delivery date. It covers manufacturing defects in the device, the power supply and the antennas.

## Not covered
Water damage, drops, unauthorised repairs and cosmetic scratches are not covered by the warranty.

## RMA process
Request an RMA (return merchandise authorisation) in the support portal with the device serial number. Once the RMA is approved we ship a replacement within 7 days, and you send the faulty unit back with the prepaid label.
""",
"api_rate_limits.md": """# API Rate Limits

## Limits per plan
Free: 60 requests per minute. Pro: 600 requests per minute. Business: 1,200 requests per minute. Enterprise: 6,000 requests per minute. Limits apply per API key.

## Best practices
Retry with exponential backoff and jitter, cache responses you read often, and send writes through the bulk endpoint.
""",
"api_errors.md": """# API Error Reference

## ERR-429-A
The per-minute rate limit was exceeded. Wait for the number of seconds in the Retry-After header, then retry.

## ERR-429-B
The daily request quota is exhausted. The quota resets at 00:00 UTC; upgrade the plan for a higher quota.

## ERR-503-C
The API is in scheduled maintenance. Retry after the maintenance window shown on the status page.

## ERR-401-D
The API key was revoked or has expired. Create a new key in Settings > API keys.
""",
"gateway_models.md": """# Edge Gateway Models

## EG-100
The EG-100 is the entry model: 2 Ethernet ports, Wi-Fi 5 and an 8 W maximum power draw. It supports up to 50 sensors.

## EG-200
The EG-200 adds LTE backup: 4 Ethernet ports, Wi-Fi 6 and a 12 W maximum power draw. It supports up to 200 sensors.

## EG-300
The EG-300 is the rugged outdoor model: an IP67 enclosure, 4 Ethernet ports, Wi-Fi 6 and an 18 W maximum power draw. It supports up to 500 sensors.
""",
"oncall_policy.md": """# On-call Policy

## Rotation
Engineers join the on-call rotation after their third month. Each shift lasts one week and hands over on Monday at 09:00 local time. Every shift has a primary and a secondary.

## Response targets
A SEV1 page must be acknowledged within 5 minutes and a SEV2 page within 30 minutes. If a SEV1 page is not acknowledged within 15 minutes, it escalates to the secondary and then to the engineering manager.

## Compensation
On-call engineers receive a stipend of $300 per week, and a day off after any night with more than two pages.
""",
"pricing.md": """# Pricing Plans

## Plans
Free costs $0 and includes 3 users and 1 GB of storage. Pro costs $49 per user per month. Business costs $99 per user per month. Enterprise pricing is custom and starts at 50 users.

## What each plan includes
Single sign-on (SAML and OIDC) is included from the Business plan upward. Audit logs and the 99.95% uptime SLA are included only in Enterprise. All paid plans include email support.

## Billing
Paying annually saves 20% compared with monthly billing. Prices exclude VAT.
""",
"sso_setup.md": """# Single Sign-On Setup

## Supported identity providers
Acme Cloud supports Okta, Microsoft Entra ID (formerly Azure AD) and Google Workspace through SAML 2.0 or OIDC.

## Configuration steps
Go to Settings > Security > Single sign-on, upload the identity provider's metadata XML, and map the email and groups attributes. Test with one account before you enforce SSO for the whole workspace.

## Troubleshooting
SSO-ERR-12 means the SAML assertion was rejected because the clocks differ by more than 5 minutes; sync the identity provider's clock with NTP. SSO-ERR-31 means the email attribute is missing from the assertion.
""",
"data_retention.md": """# Data Retention

## Event data
Raw event data is kept for 30 days on Free, 13 months on Pro and Business, and 25 months on Enterprise. Aggregated reports are kept for as long as the workspace exists.

## Deleted workspaces
When a workspace is deleted, its data is purged from primary storage after 30 days and from backups after a further 35 days.

## Erasure requests
GDPR erasure requests for one person's data are completed within 30 days of identity verification.
""",
"pto_policy.md": """# Paid Time Off

## Allowance
Full-time employees receive 25 days of paid time off per calendar year, plus public holidays. Part-time employees receive a pro-rata allowance.

## Booking
Book absences longer than three days in Workday at least two weeks ahead. Your manager approves the request.

## Carry-over
Up to 5 unused days can be carried into the next year and must be used by 31 March.
""",
"webhooks.md": """# Webhooks

## Delivery and retries
Webhook events are sent as HTTPS POST requests. If your endpoint does not return a 2xx status within 10 seconds, we retry up to 8 times with exponential backoff over 24 hours.

## Verifying signatures
Every request carries an X-Acme-Signature header: an HMAC-SHA256 of the raw body computed with your webhook secret. Compare it in constant time and reject requests older than 5 minutes.
""",
"sla.md": """# Service Level Agreement

## Uptime commitment
Pro and Business are backed by a 99.9% monthly uptime commitment. Enterprise is backed by 99.95%.

## Service credits
If monthly uptime falls below the commitment, you receive a service credit of 10% of the monthly fee; below 99.0% the credit is 25%. Credits are applied to the next invoice.

## How to claim
Open a support ticket within 30 days of the end of the affected month and include the incident dates.
""",
"account_security.md": """# Account Security

## Passwords
Password reset links expire after 60 minutes. After 5 failed sign-in attempts the account is locked for 15 minutes.

## Two-factor authentication
Workspace owners can require two-factor authentication (2FA) for every member. Authenticator apps and hardware security keys are supported; SMS codes are not.
""",
"expenses.md": """# Travel and Expenses

## Meals and hotels
Meals are reimbursed up to $75 per day while travelling. Book hotels through the travel portal; the nightly cap is $200 in most cities.

## Submitting claims
Submit receipts in Workday within 30 days of the expense. Claims over $25 without a receipt are rejected.
""",
}

In [ ]:
# The eval set: each question lists its EVIDENCE - (doc, phrase) pairs that a retrieved chunk must contain.
# Phrases are unique in the KB, so "is the evidence in the retrieved text?" works for any chunking scheme.
QUESTIONS = [
    {"q": "What does ERR-429-A mean?", "kind": "exact id", "evidence": [("api_errors.md", "per-minute rate limit")]},
    {"q": "Our integration keeps failing with ERR-401-D", "kind": "exact id", "evidence": [("api_errors.md", "revoked")]},
    {"q": "How do I fix SSO-ERR-12?", "kind": "exact id", "evidence": [("sso_setup.md", "clocks differ")]},
    {"q": "How many sensors does the EG-300 support?", "kind": "exact id", "evidence": [("gateway_models.md", "500 sensors")]},
    {"q": "How do I get my money back after buying an annual plan by mistake?", "kind": "paraphrase",
     "evidence": [("refund_policy.md", "refunded in full within 30 days")]},
    {"q": "My parcel never showed up. What should I do?", "kind": "paraphrase", "evidence": [("shipping.md", "report it within 14 days")]},
    {"q": "How many vacation days do I get a year?", "kind": "paraphrase", "evidence": [("pto_policy.md", "25 days of paid time off")]},
    {"q": "Do we get compensated for outages?", "kind": "paraphrase", "evidence": [("sla.md", "service credit")]},
    {"q": "Which gateway can be installed outdoors?", "kind": "paraphrase", "evidence": [("gateway_models.md", "rugged outdoor model")]},
    {"q": "Who gets paged when nobody acknowledges a SEV1 alert?", "kind": "keyword", "evidence": [("oncall_policy.md", "engineering manager")]},
    {"q": "How long is event data kept on the Pro plan?", "kind": "keyword", "evidence": [("data_retention.md", "13 months on Pro")]},
    {"q": "What do we get back if the platform is down more than promised?", "kind": "vague", "evidence": [("sla.md", "service credit")]},
    {"q": "What is the API rate limit on the cheapest plan that includes SSO?", "kind": "multi-hop",
     "evidence": [("pricing.md", "from the Business plan upward"), ("api_rate_limits.md", "Business: 1,200 requests")]},
    {"q": "How long do approved refunds take, and how long is the gateway warranty?", "kind": "multi-part",
     "evidence": [("refund_policy.md", "5-10 business days"), ("warranty.md", "2-year")]},
    {"q": "What is the capital of Australia?", "kind": "out-of-scope", "evidence": []},
]
IN_SCOPE = [q for q in QUESTIONS if q["evidence"]]


def has_evidence(text, doc, q):
    """True if this text (from `doc`) contains any of the question's evidence phrases."""
    return any(doc == d and p.lower() in text.lower() for d, p in q["evidence"])


print(f"{len(KB)} docs, {sum(len(t) for t in KB.values()):,} characters; "
      f"{len(QUESTIONS)} questions ({len(IN_SCOPE)} answerable, {len(QUESTIONS) - len(IN_SCOPE)} out of scope)")

In [ ]:
kb_dir = Path(tempfile.mkdtemp()) / "knowledge_base"
kb_dir.mkdir()
for name, text in KB.items():
    (kb_dir / name).write_text(text, encoding="utf-8")


def load_folder(folder):
    return {p.name: p.read_text(encoding="utf-8") for p in sorted(Path(folder).glob("*.md"))}


docs = load_folder(kb_dir)
print(f"loaded {len(docs)} files from {kb_dir.name}/:", ", ".join(list(docs)[:5]), "...")
assert docs == KB

## Step 2 - Chunk: one chunk per section, heading path prepended

Heading-aware chunking (see [notebook 01](01_chunking_strategies_and_semantic_chunking.ipynb)): the ids (`refund_policy.md#2`) double as **filename citations**.

In [ ]:
import re


def split_sections(kb):
    """One chunk per '## ' section. The heading path is prepended so every chunk says where it came from."""
    chunks = []
    for doc, text in kb.items():
        title = text.splitlines()[0].lstrip("# ").strip()
        for i, part in enumerate(re.split(r"^## ", text, flags=re.M)[1:]):
            heading, _, body = part.partition("\n")
            path = f"{title} > {heading.strip()}"
            chunks.append({"id": f"{doc}#{i}", "doc": doc, "section": path, "text": f"{path}\n{body.strip()}"})
    return chunks


def is_relevant(chunk, q):
    return has_evidence(chunk["text"], chunk["doc"], q)


CHUNKS = split_sections(KB)
print(f"{len(CHUNKS)} section chunks, avg {sum(len(c['text']) for c in CHUNKS) / len(CHUNKS):.0f} characters")
print("example:", CHUNKS[1]["id"], "->", CHUNKS[1]["text"][:90].replace("\n", " | "), "...")

## Step 3 - Embed, and guard against mixed vector spaces

Local MiniLM embeddings (384-d). The index records **which model produced it**; the query path refuses to search with a different one - a five-line check against one of the most silent RAG failures (`rg14`).

In [ ]:
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import transformers
transformers.utils.logging.disable_progress_bar()          # keep the cell output clean
transformers.logging.set_verbosity_error()
from sentence_transformers import SentenceTransformer


def load_st(name, cls=SentenceTransformer):
    """Load a small Hugging Face model: cached copy first (no network), download only if missing."""
    try:
        return cls(name, local_files_only=True)
    except Exception:
        try:
            return cls(name)                                 # first run only: ~90 MB download
        except Exception as e:
            raise RuntimeError(f"Could not load {name}: connect to the internet once so it can download.") from e


embedder = load_st("sentence-transformers/all-MiniLM-L6-v2")


def encode(texts):
    """Unit-length MiniLM embeddings (384-d), so a dot product IS the cosine similarity."""
    return embedder.encode(list(texts), normalize_embeddings=True, convert_to_numpy=True)


print("embedder: all-MiniLM-L6-v2 |", embedder.get_embedding_dimension(), "dims | max",
      embedder.max_seq_length, "tokens per text (the rest is silently cut off)")

In [ ]:
INDEX = {
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "vectors": encode([c["text"] for c in CHUNKS]),
    "chunks": CHUNKS,
}


def embed_query(text, model_name="sentence-transformers/all-MiniLM-L6-v2"):
    if model_name != INDEX["embedding_model"]:
        raise ValueError(f"query model {model_name!r} != index model {INDEX['embedding_model']!r}: "
                         "vectors from different models are not comparable - re-embed the corpus instead")
    return encode([text])[0]


print("index:", INDEX["vectors"].shape, "built with", INDEX["embedding_model"])
try:
    embed_query("refund window", model_name=EMBED_MODEL)            # e.g. text-embedding-3-small
except ValueError as e:
    print("guard works ->", str(e)[:110], "...")

**The API route.** In production you would often use a hosted embedding model instead. `llm_setup.embed()` calls `client.embeddings.create` on the configured provider (`text-embedding-3-small` with an OpenAI key, `nomic-embed-text` with Ollama, a hashed bag-of-words offline). Its vectors live in a *different space with a different dimension* - you would re-embed the whole corpus with it, never mix:

In [ ]:
from llm_setup import embed

api_vecs = np.array(embed(["How do I get a refund?", "Refund Policy: annual plans can be refunded within 30 days."]))
print(f"{EMBED_MODEL} via {PROVIDER}: {api_vecs.shape[1]} dims (MiniLM: {INDEX['vectors'].shape[1]}) | "
      f"cosine of the pair = {api_vecs[0] @ api_vecs[1] / np.linalg.norm(api_vecs[0]) / np.linalg.norm(api_vecs[1]):.3f}")

## Step 4 - Index and retrieve (hybrid)

A NumPy matrix is the vector index (exact search is fine for 42 chunks - and up to ~100k), BM25 is the lexical index, RRF fuses them ([notebook 04](04_hybrid_search_and_rrf.ipynb)).

In [ ]:
STOPWORDS = set("""a an the is are was were be been of to in on for with and or but if then so as at by from this that
it its i you we my our your do does did can how what who when which why there their they me""".split())


def tokenize(text):
    return [t for t in re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower()) if t not in STOPWORDS]


BM25_INDEX = BM25Okapi([tokenize(c["text"]) for c in CHUNKS])


def retrieve(question, k=4, depth=10):
    """Hybrid retrieval. Returns the top-k chunks plus the raw signals the confidence gate needs."""
    b = BM25_INDEX.get_scores(tokenize(question))
    lex = [i for i in np.argsort(-b, kind="stable") if b[i] > 0][:depth]
    cos = INDEX["vectors"] @ embed_query(question)
    dense = list(np.argsort(-cos, kind="stable")[:depth])
    fused = {}
    for lane in (lex, dense):
        for rank, i in enumerate(lane, start=1):
            fused[i] = fused.get(i, 0.0) + 1 / (60 + rank)
    top = sorted(fused, key=lambda i: -fused[i])[:k]
    return [CHUNKS[i] for i in top], {"lexical_matches": len(lex), "max_cosine": float(cos.max())}


hits, signals = retrieve("How long do approved refunds take?")
print(signals)
for c in hits:
    print("  ", c["id"], "|", c["text"].replace("\n", " | ")[:80])

## Step 5 - The confidence gate (empty / low-confidence retrieval)

Passing irrelevant context to a model and hoping is the biggest single source of RAG hallucination. Decide **before** calling the model: if no chunk shares a term with the question *and* the best cosine is below a floor, refuse. The floor must be calibrated on labelled questions - look at the distribution first:

In [ ]:
rows = []
for q in QUESTIONS:
    _, s = retrieve(q["q"])
    rows.append({"kind": q["kind"], "question": q["q"][:55], **s})
sig = pd.DataFrame(rows).sort_values("max_cosine")
print(sig.to_string(index=False))

MIN_COSINE = 0.15      # between the out-of-scope question and the weakest answerable one - calibrate on YOUR data
in_scope_min = sig[sig["kind"] != "out-of-scope"]["max_cosine"].min()
oos_max = sig[sig["kind"] == "out-of-scope"]["max_cosine"].max()
assert oos_max < MIN_COSINE < in_scope_min, "the floor must separate the labelled cases"


def confident(signals):
    return signals["lexical_matches"] > 0 or signals["max_cosine"] >= MIN_COSINE

Only the out-of-scope question has zero lexical matches *and* a tiny cosine. Note how close the vague in-scope question sits to the floor - raw cosine is not calibrated (`rg49`), which is why real systems add the reranker score or the rank-1 vs rank-5 gap and recalibrate whenever the corpus or the embedder changes.

## Step 6 - The grounded prompt

Rules in the system message; the question and numbered, id-tagged context in the user message. The id is what the model must cite.

In [ ]:
SYSTEM_PROMPT = (
    "You answer questions for Acme Cloud's support team. Use only the context passages. "
    "After each claim, cite the passage id in square brackets, for example [doc.md#0]. "
    "If the context does not contain the answer, reply exactly: I don't know."
)


def build_messages(question, chunks):
    context = "\n".join(f"[{c['id']}] {c['text'].replace(chr(10), ' ')}" for c in chunks)
    user = f"Question: {question}\n\nContext:\n{context}"
    return [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user}]


question = "How long do approved refunds take?"
hits, _ = retrieve(question)
messages = build_messages(question, hits)
print("=== EXACT PROMPT SENT ===")
for m in messages:
    print(f"--- {m['role']} ---\n{m['content']}")

## Step 7 - Generate, then parse and verify the citations

In [ ]:
CITATION = re.compile(r"\[([\w.\-]+\.md#\d+)\]")


def generate(messages):
    resp = client.chat.completions.create(model=MODEL, messages=messages, temperature=0, max_tokens=300)
    return resp.choices[0].message.content


def verify_citations(answer, chunks):
    cited = CITATION.findall(answer)
    sent = {c["id"] for c in chunks}
    return {"cited": cited, "all_cited_were_retrieved": set(cited) <= sent,
            "unknown_ids": sorted(set(cited) - sent)}


answer = generate(messages)
check = verify_citations(answer, hits)
print(f"provider: {PROVIDER}\nanswer  : {answer}\ncheck   : {check}")
assert check["all_cited_were_retrieved"]

**What the offline stand-in does here:** it is an *extractive* answerer - it quotes the context sentence(s) that best overlap the question and appends that passage's `[id]` tag. That is enough to exercise the whole pipeline honestly (retrieval, prompt, parsing, verification, evaluation). A real model (`gpt-4.1-mini`, or `qwen2.5:3b` via Ollama) writes a fluent sentence such as *"Approved refunds reach the original payment method within 5-10 business days; bank transfers can take up to 15 [refund_policy.md#2]."* - same code, better prose.

## Step 8 - The whole pipeline, including the refusal path

In [ ]:
def answer_question(question, k=4, use_gate=True):
    chunks, signals = retrieve(question, k=k)
    if use_gate and not confident(signals):
        return {"answer": "I don't know - nothing in the knowledge base matches that question.",
                "chunks": [], "refused_before_llm": True, "signals": signals}
    text = generate(build_messages(question, chunks))
    return {"answer": text, "chunks": chunks, "refused_before_llm": False, "signals": signals,
            **verify_citations(text, chunks)}


oos = "What is the capital of Australia?"
gated = answer_question(oos)
print("with the gate   :", gated["answer"], "| LLM called:", not gated["refused_before_llm"])
ungated = answer_question(oos, use_gate=False)
print("without the gate:", ungated["answer"])
assert gated["refused_before_llm"] and "don't know" in ungated["answer"].lower()

Two layers of defence. The **gate** refuses without spending a model call (and can log the miss as a corpus gap). Without it, the **prompt rule** still makes the model say "I don't know" here - but a prompt rule is a request, not a control: models do answer from memory when handed irrelevant context, which is exactly what the gate prevents. Real systems keep both.

## Step 9 - Evaluate on the labelled questions

Two layers, graded separately (`rg11`): **retrieval** (was the evidence in the chunks we sent?) and **generation** (did the answer cite a chunk that holds the evidence, and only chunks we sent?).

In [ ]:
rows = []
for q in QUESTIONS:
    out = answer_question(q["q"])
    retrieved_ok = any(is_relevant(c, q) for c in out["chunks"])
    cited_chunks = [c for c in out["chunks"] if c["id"] in out.get("cited", [])]
    rows.append({
        "kind": q["kind"],
        "question": q["q"][:48],
        "evidence retrieved": retrieved_ok if q["evidence"] else None,
        "cites evidence": any(is_relevant(c, q) for c in cited_chunks) if q["evidence"] else None,
        "citations valid": out.get("all_cited_were_retrieved", True),
        "refused": out["refused_before_llm"] or "don't know" in out["answer"].lower(),
        "answer": out["answer"][:60],
    })
ev = pd.DataFrame(rows)
print(ev.drop(columns="answer").to_string(index=False))

answerable = ev[ev["kind"] != "out-of-scope"]
print(f"\nretrieval hit@4      : {answerable['evidence retrieved'].mean():.2f}")
print(f"answers citing evidence: {answerable['cites evidence'].mean():.2f}")
print(f"citations all valid  : {ev['citations valid'].mean():.2f}")
print(f"out-of-scope refused : {bool(ev[ev['kind'] == 'out-of-scope']['refused'].all())}")
assert ev["citations valid"].all() and ev[ev["kind"] == "out-of-scope"]["refused"].all()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.4))
metrics = {"evidence retrieved\n(retrieval)": answerable["evidence retrieved"].mean(),
           "answer cites evidence\n(generation)": answerable["cites evidence"].mean(),
           "citations valid\n(no invented ids)": ev["citations valid"].mean()}
ax.bar(list(metrics), list(metrics.values()), color=["#3b5bdb", "#e8590c", "#2b8a3e"])
for i, v in enumerate(metrics.values()):
    ax.text(i, v + 0.02, f"{v:.2f}", ha="center")
ax.set_ylim(0, 1.15)
ax.set_title(f"End-to-end eval on {len(answerable)} answerable questions (provider: {PROVIDER})")
plt.tight_layout()
plt.show()

Read the gap between the first two bars. Where the evidence was retrieved but the answer does not cite it, the **generator** is the problem. Offline you can see both flavours: for the parcel question the extractive stand-in latched onto a sentence that shares more words with the question, and for "Do we get compensated for outages?" it found no shared words at all and said "I don't know" - even though the SLA chunk was in the prompt. A real model reads meaning and gets both. Where the evidence was never retrieved (the vague SLA question), no model could have answered - that is a **retrieval** ticket, fixed in [notebook 07](07_advanced_rag_query_rewriting_hyde_multiquery.ipynb). Re-run this cell with a real provider and the generation bar is the one that moves; the retrieval bar does not change, because retrieval does not depend on the LLM.

## Failure modes and their fixes (`rg23`)

Debug in this order - the first three need no LLM: *is the chunk in the index? in the top k? in the prompt? did the model use it?*

| Failure | Symptom | How to detect | Fix |
|---|---|---|---|
| **Bad chunking** | the answer is split across chunks, or a chunk is a lone heading | read the retrieved chunks; hit-rate vs chunk-size sweep | heading-aware chunks with the heading path; sensible size + overlap; parent-child |
| **Missing document** | confident nonsense or "I don't know" on a real question | evidence not in the index at all; empty-retrieval rate | ingestion fix; freshness SLO; log gate refusals as corpus gaps |
| **Wrong doc ranked first** | answer cites a near-miss (ERR-429-B for ERR-429-A) | evidence at rank 2-10; MRR drops | hybrid (BM25 for identifiers), reranker, metadata filters |
| **Relevant doc not retrieved at all** | vague or paraphrased questions fail | recall@50 low | query rewriting / multi-query / HyDE; better embedder |
| **Context overflow** | late chunks silently ignored or truncated | count prompt tokens vs the window; lost-in-the-middle | fewer, better chunks (rerank, k = 3-5); trimming policy with headroom |
| **Model ignores the context** | answer contradicts the cited chunk, or answers from memory | faithfulness check; answer not supported by any sent chunk | confidence gate; stricter prompt; citation verification; a stronger model |
| **Fabricated citation** | `[pricing.md#9]` that was never sent | set membership check on cited ids (Step 7) | reject / retry once with the validator's complaint, else refuse |

## Try it yourself

**Exercise 1.** Retrieval depth: run the eval with `k = 2` and `k = 8`. What happens to retrieval hit-rate - and to the prompt size you pay for?

In [ ]:
# Solution — try it yourself first, then run this
for k in (2, 4, 8):
    hit = np.mean([any(is_relevant(c, q) for c in retrieve(q["q"], k=k)[0]) for q in IN_SCOPE])
    chars = np.mean([len(build_messages(q["q"], retrieve(q["q"], k=k)[0])[1]["content"]) for q in IN_SCOPE])
    print(f"k={k}: retrieval hit = {hit:.2f}, avg user-message size = {chars:.0f} characters")

**Exercise 2.** Make the citation verifier catch a fabricated id and a correct one in the same answer.

In [ ]:
# Solution — try it yourself first, then run this
fake = "Refunds take 5-10 business days [refund_policy.md#2]. Enterprise gets a discount [pricing.md#9]."
print(verify_citations(fake, retrieve("How long do approved refunds take?")[0]))

**Exercise 3.** Freshness: the refund window changes from 30 to 45 days. Update the file, re-index **only the changed chunk** (compare section texts), and ask again.

In [ ]:
# Solution — try it yourself first, then run this
(kb_dir / "refund_policy.md").write_text(KB["refund_policy.md"].replace("within 30 days", "within 45 days"), encoding="utf-8")
new_by_id = {c["id"]: c for c in split_sections(load_folder(kb_dir))}
changed = [i for i, c in enumerate(CHUNKS) if new_by_id[c["id"]]["text"] != c["text"]]     # match by id, not position
for i in changed:
    CHUNKS[i] = new_by_id[CHUNKS[i]["id"]]
    INDEX["vectors"][i] = encode([CHUNKS[i]["text"]])[0]
assert len(changed) == 1
BM25_INDEX = BM25Okapi([tokenize(c["text"]) for c in CHUNKS])       # BM25 is cheap to rebuild at this size
print("re-embedded chunks:", [CHUNKS[i]["id"] for i in changed])
print(answer_question("How do I get my money back after buying an annual plan by mistake?")["answer"])

## Say it in an interview

- **30 seconds (`rg00`):** "Python. At index time: load the folder, split by headings with the heading path prepended, embed with one model, store vectors plus a BM25 index, and record the embedding model id. At query time: hybrid retrieval fused with RRF, a confidence gate that refuses before calling the model when nothing matches, a prompt that says use only this context, cite the passage id after each claim, say I don't know otherwise - then I verify every cited id was actually retrieved. Framework-wise, LangChain gives the same shape with swappable parts."
- **What I add second:** a 50-100 question eval set with labelled evidence; a cross-encoder reranker; metadata / permission filters applied before retrieval; incremental re-indexing keyed on content hash.
- **Numbers:** k = 3-5 chunks after reranking; chunks ~200-800 tokens; the gate threshold is calibrated, never guessed.
- **Traps:** blaming the model before checking retrieval (`rg55`: most "hallucinations" are retrieval tickets); an empty context passed to the model; trusting cited ids; mixing embedding models; no out-of-scope questions in the eval set, so refusals are never tested.
- **Follow-ups:** "how do you know it works in production?" (frozen eval on every change, empty-retrieval and citation-failure rates, follow-up-question rate, weekly human review); "what changes at 50M documents?" (ANN + PQ, sharding, filters first, caching).

## Next

- [Advanced RAG: query rewriting, HyDE, multi-query](07_advanced_rag_query_rewriting_hyde_multiquery.ipynb) - fix the retrieval failures found above.
- [RAG evaluation: retrieval and faithfulness](../12_evaluation/03_rag_evaluation_retrieval_and_faithfulness.ipynb) · [LangChain RAG chain end to end](../13_langchain/05_rag_chain_end_to_end.ipynb) · [RAG skeleton coding round](../17_interview_coding_rounds/01_rag_skeleton.ipynb)
- [Reranking](05_reranking.ipynb) · [Guardrails, PII and prompt injection](../16_production/07_guardrails_pii_and_prompt_injection.ipynb)
- Theory: [genai_flow course](../../genai_flow/index.html) · [AI system design concepts](../../ai_system_design_concepts/index.html) · [interview bank](../../ai_interview_prep/index.html)